In [1]:
import pandas as pd

clean_df = pd.read_csv(
    "../data/processed/clean_comments.csv"
)

print("Shape:", clean_df.shape)
clean_df.head()

Shape: (110336, 8)


,comment_id,entry_id,user,source,url,timestamp,comment,comment_clean
0,e/624ca9226b6526ebdb69f9b46df482c7/c/32c6bf5bc...,e/624ca9226b6526ebdb69f9b46df482c7,guardianuk,NaN,NaN,2010-08-06 14:45:07,Reel Review video: Catherine Shoard defends Kn...,Reel Review video: Catherine Shoard defends Kn...
1,e/967b4db48fa74021b24ccbc93c55a61c/c/57905e983...,e/967b4db48fa74021b24ccbc93c55a61c,massoptimization,Bookmarklet,http://friendfeed.com/share/bookmarklet,2010-08-06 15:06:44,Article by at 2010-08-06 09:42:14 Categoriz...,Article by at 2010-08-06 09:42:14 Categorized ...
2,e/79e585effdf640e988539e2dd68c2c6d/c/853946596...,e/79e585effdf640e988539e2dd68c2c6d,sheribabysph,Blip.fm,http://blip.fm/,2010-08-06 15:06:44,Happy Friday! @Skyblue101 and @TropicsZ4: Matc...,Happy Friday! @Skyblue101 and @TropicsZ4: Matc...
3,e/01e9a149fff85ff948972896145d3d65/c/e87986e73...,e/01e9a149fff85ff948972896145d3d65,inhabitat24,NaN,NaN,2010-08-06 15:06:27,Yesterday the enviously green city of Portland...,Yesterday the enviously green city of Portland...
4,e/01e9a149fff85ff948972896145d3d65/c/c70184cfd...,e/01e9a149fff85ff948972896145d3d65,inhabitat24,NaN,NaN,2010-08-06 15:06:27,Yesterday the enviously green city of Portland...,Yesterday the enviously green city of Portland...


In [2]:
comment_counts = (
    clean_df
    .groupby("entry_id")
    .size()
    .reset_index(name="CommentCount")
)

print("Unique entries:", len(comment_counts))
comment_counts.head()

Unique entries: 91071


,entry_id,CommentCount
0,e/0000b70d9f3b1d0a7e78bc2ce09af1a2,1
1,e/0000d26daa876fa3165f8808656a30d9,1
2,e/00025446898d8328d8272cce96072696,1
3,e/0004a4a67a3446d79be082181f50c01b,1
4,e/0006970b646d4530991c745db20055b8,1


In [3]:
likes_file = "../data/raw/likes_extracted/likes.csv"

In [4]:
comment_entry_ids = set(
    clean_df["entry_id"]
    .dropna()
    .unique()
)

like_counts_dict = {}

for chunk in pd.read_csv(
    likes_file,
    sep="\t",
    header=None,
    names=[
        "user",
        "entry_id",
        "like_timestamp"
    ],
    chunksize=10000,
    dtype=str
):

    # Keep likes only for entries present in our clean comments
    chunk = chunk[
        chunk["entry_id"].isin(comment_entry_ids)
    ]

    counts = (
        chunk["entry_id"]
        .value_counts()
    )

    for entry_id, count in counts.items():
        like_counts_dict[entry_id] = (
            like_counts_dict.get(entry_id, 0)
            + count
        )

In [5]:
like_counts = pd.DataFrame(
    list(like_counts_dict.items()),
    columns=[
        "entry_id",
        "like_count"
    ]
)

print(
    "Entries with at least one like:",
    len(like_counts)
)

like_counts.head()

Entries with at least one like: 6349


,entry_id,like_count
0,e/a90e8710d1f143ff86654ce02ded7896,34
1,e/64f636cabf21440b81d73fc220766210,28
2,e/cadb4ad14e534afa84329de8e2478211,32
3,e/d191dcb838e449debfa7627a630dc0c7,13
4,e/87ee296aa23d4bc9914a2f408db68925,3


In [6]:
engagement_df = comment_counts.merge(
    like_counts,
    on="entry_id",
    how="left"
)

engagement_df["like_count"] = (
    engagement_df["like_count"]
    .fillna(0)
    .astype(int)
)

engagement_df.head()

,entry_id,CommentCount,like_count
0,e/0000b70d9f3b1d0a7e78bc2ce09af1a2,1,0
1,e/0000d26daa876fa3165f8808656a30d9,1,0
2,e/00025446898d8328d8272cce96072696,1,0
3,e/0004a4a67a3446d79be082181f50c01b,1,0
4,e/0006970b646d4530991c745db20055b8,1,1


In [7]:
print("Shape:", engagement_df.shape)

print(
    "Entries with likes:",
    (engagement_df["like_count"] > 0).sum()
)

print(
    "Entries without likes:",
    (engagement_df["like_count"] == 0).sum()
)

Shape: (91071, 3)
Entries with likes: 6349
Entries without likes: 84722


In [8]:
engagement_df[
    ["like_count", "CommentCount"]
].describe()

,like_count,CommentCount
count,91071.000000,91071.000000
mean,0.196023,1.211538
std,3.104492,5.023452
min,0.000000,1.000000
25%,0.000000,1.000000
50%,0.000000,1.000000
75%,0.000000,1.000000
max,404.000000,1075.000000


In [9]:
output_path = "../data/processed/engagement_data.csv"

engagement_df.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)
print("Shape:", engagement_df.shape)

Saved: ../data/processed/engagement_data.csv
Shape: (91071, 3)


In [10]:
saved_engagement = pd.read_csv(
    "../data/processed/engagement_data.csv"
)

saved_engagement.head()

,entry_id,CommentCount,like_count
0,e/0000b70d9f3b1d0a7e78bc2ce09af1a2,1,0
1,e/0000d26daa876fa3165f8808656a30d9,1,0
2,e/00025446898d8328d8272cce96072696,1,0
3,e/0004a4a67a3446d79be082181f50c01b,1,0
4,e/0006970b646d4530991c745db20055b8,1,1
